# GA/SA 特徵篩選 Best Integrated Notebook（中文版）

這份是整合後的最佳版本：以我們原本的 all-in-one 可重跑流程為主，吸收兩份 Colab 的優點，並修正不符合 Word 說明書的地方。

整合原則：

- 採用 Colab B 的報告章節架構與中文圖表呈現方式。
- 採用 Colab A/B 的 baseline 思路：Random Search、All Features、Top-K MI、GA/SA Jaccard overlap。
- 保留我們版的自動化與完整輸出：multi-seed、aggregate mean/std、完整 train_transaction final 5-fold CV、中文 HTML。
- 修正風險：不使用 Colab A 的 debug 預設、不使用 A 的 100,000 筆 final sample、不使用 Colab B 的硬編碼結果、不使用 B 後段的 SA alpha=0.999。

正式交作業請直接從上到下 Run All。仍然需要的外部資料只有 `data/raw/train_transaction.csv`。

這個 `.ipynb` 以 ASCII JSON 儲存，Jupyter/VS Code 顯示中文正常，也能避免 Windows 編碼造成 raw notebook 亂碼。


## 建議執行方式

這份是「加速整合版」，適合你現在這台電腦先跑出完整比較結果。

1. 把本 notebook 放在 `GA_SA_feature_selection_project` 資料夾內。
2. 確認 `data/raw/train_transaction.csv` 存在。
3. 從上到下 `Run All`。預設會執行 `word_practical` multi-seed。
4. 執行時 cell 輸出會顯示 seed、lambda、Random Search、GA generation、SA iteration、final CV 進度。
5. 每完成一個 lambda，會先寫出 `partial_metrics.csv`、`partial_convergence.csv`、`partial_selected_features.json`，方便中途檢查進度。
6. 跑完後最後一格會列出最新 `aggregate_*` 資料夾與中文 HTML 報告路徑。

如果只是確認環境，請先把主要開關改成：

```python
RUN_SMOKE = True
RUN_MULTI_SEED = False
```

建議正式跑法：

```python
RUN_MULTI_SEED = True
REAL_PRESET = "word_practical"
FINAL_EVAL_FULL_TRAINING_SET = True
SEARCH_DEVICE = "cpu"
FINAL_DEVICE = "auto"
```

preset 差異：

- `word_practical`：保留 20,000 proxy rows、MI Top 150、四組 lambda、完整 train_transaction.csv final 5-fold；搜尋階段改用 holdout 並減少 GA/SA/Random 次數，速度最快。
- `word_balanced`：搜尋階段用 3-fold，GA/SA 次數介於 practical 與 strict 之間。
- `word_strict`：完全照 Word 表格的嚴格版，最慢；如果老師要求搜尋 fitness 也必須 5-fold，就用這個。



## Word 要求對照

這份 notebook 的正式 preset 是 `word_strict`，對應 Word 說明書要求：

- 資料集：IEEE-CIS Fraud Detection，只使用 `train_transaction.csv`，不合併 identity 表。
- Proxy Dataset：GA/SA 搜尋使用 stratified sampling 的 20,000 筆。
- Final Evaluation：搜尋完成後，將候選特徵子集套用到完整 `train_transaction.csv`，使用 5-fold CV 評估 final AUC。
- 前處理：缺失值填補、類別編碼、VarianceThreshold、Mutual Information Top 150。
- Fitness：`AUC - λ × selected_features / total_features`。
- λ：`0.0, 0.05, 0.10, 0.20`。
- GA：population 40、generation 60、tournament k=3、one-point crossover 0.85、mutation 1/150、elitism 2。
- SA：initial temperature calibration 100 moves、geometric cooling alpha 0.98、T_min 1e-4、iterations 5000、neighbor flip 1-3 bits。
- Baselines：Random Search（2400 evaluations，對齊 GA 40×60）、All Search Features、Top-K MI。
- 穩定性：多 seed 自動跑完後輸出 mean/std。
- 特徵重疊：輸出 GA/SA Jaccard similarity。
- 視覺化：收斂曲線、MI 特徵重要性/選取、Pareto AUC vs 特徵數、λ 敏感度。


In [ ]:
# 這格會檢查套件、設定 UTF-8、建立基本路徑。沒有缺套件時不會安裝任何東西。
import base64
import hashlib
import html
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import textwrap
import warnings
from dataclasses import asdict, dataclass, replace
from datetime import datetime
from math import exp, log
from pathlib import Path
from time import perf_counter
from typing import Callable, Iterable, Literal

os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
}
_missing = [pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("缺少套件，正在安裝：", ", ".join(_missing), flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore", category=UserWarning)


def configure_chinese_matplotlib() -> None:
    """Use a Traditional Chinese capable font when one is available."""
    candidates = [
        "Microsoft JhengHei",
        "Noto Sans CJK TC",
        "Noto Sans TC",
        "PingFang TC",
        "SimHei",
        "Arial Unicode MS",
        "DejaVu Sans",
    ]
    installed = {font.name for font in fm.fontManager.ttflist}
    chosen = next((name for name in candidates if name in installed), "DejaVu Sans")
    plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    print("matplotlib 字型：", chosen)


configure_chinese_matplotlib()

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train_transaction.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("工作資料夾：", PROJECT_ROOT)
print("資料檔：", DATA_PATH)
print("結果資料夾：", RESULTS_DIR)

## 整合來源與採用決策

| 來源 | 採用內容 | 不採用/修正內容 |
|---|---|---|
| Colab A | Top-K MI K=GA/K=SA、Jaccard overlap、lambda sweep 的思路 | 不採用 `DEBUG_MODE=True` 的少量設定；不採用 100,000 筆 final sample 當正式 final evaluation |
| Colab B | 中文章節架構、完整資料 final evaluation、四張圖命名、中文字型設定、穩定性分析呈現 | 不採用硬編碼 seed=42 結果；不採用後段 SA `alpha=0.999` |
| 我們原本 all-in-one | 全流程自動化、cache、multi-seed aggregate、完整 train_transaction final 5-fold、中文 HTML | 補強報告架構與中文圖表文字，並把 Random Search 改成 2400 次以對齊 GA evaluations |

最終判斷：程式執行以本 notebook 為準；Colab A/B 只作為架構與呈現方式參考。


## 報告章節建議

跑完本 notebook 後，可以依照下列順序撰寫 Word 報告：

1. 題目背景與研究動機：高維特徵篩選、fraud detection、不平衡分類、GA/SA 搜尋。
2. 資料集說明：IEEE-CIS Fraud Detection、`train_transaction.csv`、590,540 筆、isFraud 比例。
3. 前處理策略：缺失值、類別編碼、VarianceThreshold、Mutual Information Top 150。
4. Fitness 設計：AUC-ROC 與特徵數懲罰。
5. GA 方法：chromosome、selection、crossover、mutation、elitism、termination。
6. SA 方法：state、neighbor move、temperature calibration、cooling、acceptance rule。
7. Baseline：Random Search、All Search Features、Top-K MI。
8. 實驗設定：proxy 20,000、final full train 5-fold、lambda、seeds、XGBoost。
9. 結果比較：final AUC、fitness、selected_count、seconds。
10. 收斂分析：GA/SA convergence。
11. Pareto 分析：AUC vs selected features。
12. λ 敏感度：AUC 與特徵數 trade-off。
13. 特徵重疊：GA/SA Jaccard 與重要特徵。
14. 穩定性分析：multi-seed mean/std。
15. 結論：哪個方法較好、特徵數與 AUC 的取捨、限制與未來改進。


In [ ]:
from dataclasses import dataclass, replace
from typing import Literal


PresetName = Literal["smoke", "quick", "standard"]


@dataclass(frozen=True)
class ExperimentConfig:
    preset: PresetName
    proxy_rows: int
    search_features: int
    lambdas: tuple[float, ...]
    random_trials: int
    ga_population: int
    ga_generations: int
    ga_elites: int
    ga_tournament_k: int
    ga_crossover_rate: float
    ga_mutation_rate: float | None
    sa_iterations: int
    sa_log_interval: int
    sa_alpha: float
    sa_t_min: float
    sa_calibration_moves: int
    inner_cv_folds: int
    final_cv_folds: int
    test_size: float
    xgb_estimators_inner: int
    xgb_estimators_final: int
    xgb_max_depth: int
    xgb_learning_rate: float
    random_seed: int


PRESETS: dict[PresetName, ExperimentConfig] = {
    "smoke": ExperimentConfig(
        preset="smoke",
        proxy_rows=3_000,
        search_features=30,
        lambdas=(0.05,),
        random_trials=20,
        ga_population=8,
        ga_generations=5,
        ga_elites=2,
        ga_tournament_k=3,
        ga_crossover_rate=0.85,
        ga_mutation_rate=None,
        sa_iterations=80,
        sa_log_interval=10,
        sa_alpha=0.98,
        sa_t_min=1e-4,
        sa_calibration_moves=10,
        inner_cv_folds=1,
        final_cv_folds=3,
        test_size=0.2,
        xgb_estimators_inner=30,
        xgb_estimators_final=60,
        xgb_max_depth=3,
        xgb_learning_rate=0.08,
        random_seed=42,
    ),
    "quick": ExperimentConfig(
        preset="quick",
        proxy_rows=20_000,
        search_features=80,
        lambdas=(0.05, 0.10),
        random_trials=120,
        ga_population=16,
        ga_generations=18,
        ga_elites=2,
        ga_tournament_k=3,
        ga_crossover_rate=0.85,
        ga_mutation_rate=None,
        sa_iterations=600,
        sa_log_interval=50,
        sa_alpha=0.98,
        sa_t_min=1e-4,
        sa_calibration_moves=25,
        inner_cv_folds=1,
        final_cv_folds=3,
        test_size=0.2,
        xgb_estimators_inner=60,
        xgb_estimators_final=120,
        xgb_max_depth=4,
        xgb_learning_rate=0.08,
        random_seed=42,
    ),
    "standard": ExperimentConfig(
        preset="standard",
        proxy_rows=20_000,
        search_features=120,
        lambdas=(0.0, 0.05, 0.10, 0.20),
        random_trials=300,
        ga_population=24,
        ga_generations=30,
        ga_elites=2,
        ga_tournament_k=3,
        ga_crossover_rate=0.85,
        ga_mutation_rate=None,
        sa_iterations=1_500,
        sa_log_interval=100,
        sa_alpha=0.98,
        sa_t_min=1e-4,
        sa_calibration_moves=40,
        inner_cv_folds=1,
        final_cv_folds=5,
        test_size=0.2,
        xgb_estimators_inner=80,
        xgb_estimators_final=160,
        xgb_max_depth=4,
        xgb_learning_rate=0.08,
        random_seed=42,
    ),
}


def get_config(name: PresetName, **overrides: object) -> ExperimentConfig:
    config = PRESETS[name]
    clean = {key: value for key, value in overrides.items() if value is not None}
    return replace(config, **clean) if clean else config


In [ ]:
# Word 相關 preset：
# word_strict  = 完全照 Word 表格，最慢。
# word_balanced = 保留 Word 的資料量/特徵數/lambda/final 5-fold，但搜尋階段用較少次數與 3-fold。
# word_practical = 建議先跑，保留 final 完整 5-fold，但搜尋階段改 holdout，大幅加速。
WORD_STRICT_CONFIG = replace(
    PRESETS["standard"],
    preset="word_strict",
    proxy_rows=20_000,
    search_features=150,
    lambdas=(0.0, 0.05, 0.10, 0.20),
    random_trials=2_400,
    ga_population=40,
    ga_generations=60,
    ga_elites=2,
    ga_tournament_k=3,
    ga_crossover_rate=0.85,
    ga_mutation_rate=1.0 / 150.0,
    sa_iterations=5_000,
    sa_log_interval=100,
    sa_alpha=0.98,
    sa_t_min=1e-4,
    sa_calibration_moves=100,
    inner_cv_folds=5,
    final_cv_folds=5,
    test_size=0.2,
    xgb_estimators_inner=80,
    xgb_estimators_final=160,
    xgb_max_depth=4,
    xgb_learning_rate=0.08,
)

WORD_BALANCED_CONFIG = replace(
    WORD_STRICT_CONFIG,
    preset="word_balanced",
    random_trials=1_200,
    ga_population=30,
    ga_generations=40,
    sa_iterations=2_500,
    sa_calibration_moves=60,
    inner_cv_folds=3,
    xgb_estimators_inner=60,
)

WORD_PRACTICAL_CONFIG = replace(
    WORD_STRICT_CONFIG,
    preset="word_practical",
    random_trials=720,
    ga_population=24,
    ga_generations=30,
    sa_iterations=1_500,
    sa_calibration_moves=40,
    inner_cv_folds=1,
    xgb_estimators_inner=60,
)

NOTEBOOK_PRESETS = {
    "word_strict": WORD_STRICT_CONFIG,
    "word_balanced": WORD_BALANCED_CONFIG,
    "word_practical": WORD_PRACTICAL_CONFIG,
}


def get_notebook_config(name: str, **overrides: object) -> ExperimentConfig:
    config = NOTEBOOK_PRESETS[name] if name in NOTEBOOK_PRESETS else get_config(name)  # type: ignore[arg-type]
    clean = {key: value for key, value in overrides.items() if value is not None}
    return replace(config, **clean) if clean else config


In [ ]:
import hashlib
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.model_selection import train_test_split


@dataclass
class PreparedData:
    X: np.ndarray
    y: np.ndarray
    feature_names: list[str]
    selected_from_original: list[str]
    source: str
    rows: int
    original_features: int


CACHE_VERSION = "preprocess-v2-mi-desc"


def load_synthetic(proxy_rows: int, random_seed: int) -> tuple[pd.DataFrame, pd.Series]:
    n_features = 150
    X, y = make_classification(
        n_samples=proxy_rows,
        n_features=n_features,
        n_informative=35,
        n_redundant=20,
        n_repeated=0,
        n_clusters_per_class=2,
        weights=[0.965, 0.035],
        flip_y=0.01,
        class_sep=1.1,
        random_state=random_seed,
    )
    names = [f"f{i:03d}" for i in range(n_features)]
    return pd.DataFrame(X, columns=names), pd.Series(y, name="isFraud")


def load_ieee_transaction(path: Path, proxy_rows: int, random_seed: int) -> tuple[pd.DataFrame, pd.Series]:
    if not path.exists():
        raise FileNotFoundError(
            f"找不到資料檔：{path}\n"
            "請先把 Kaggle IEEE-CIS 的 train_transaction.csv 放到 data/raw/。"
        )

    df = pd.read_csv(path, low_memory=False)
    if "isFraud" not in df.columns:
        raise ValueError("train_transaction.csv 中找不到 isFraud 欄位。")

    if proxy_rows and len(df) > proxy_rows:
        _, df = train_test_split(
            df,
            test_size=proxy_rows,
            stratify=df["isFraud"],
            random_state=random_seed,
        )

    y = df.pop("isFraud").astype("int8")
    for col in ("TransactionID",):
        if col in df.columns:
            df = df.drop(columns=[col])
    return df.reset_index(drop=True), y.reset_index(drop=True)


def prepare_ieee_transaction_with_cache(
    path: Path,
    proxy_rows: int,
    search_features: int,
    random_seed: int,
    cache_dir: Path,
) -> tuple[PreparedData, bool, Path]:
    if not path.exists():
        raise FileNotFoundError(
            f"找不到資料檔：{path}\n"
            "請先把 Kaggle IEEE-CIS 的 train_transaction.csv 放到 data/raw/。"
        )

    cache_dir.mkdir(parents=True, exist_ok=True)
    key = _cache_key(path, proxy_rows, search_features, random_seed)
    npz_path = cache_dir / f"{key}.npz"
    meta_path = cache_dir / f"{key}.json"

    if npz_path.exists() and meta_path.exists():
        return _load_prepared_cache(npz_path, meta_path), True, npz_path

    X_raw, y = load_ieee_transaction(path, proxy_rows, random_seed)
    data = preprocess_and_select(
        X_raw=X_raw,
        y=y,
        search_features=search_features,
        random_seed=random_seed,
        source=str(path),
    )
    _save_prepared_cache(data, npz_path, meta_path, path, proxy_rows, search_features, random_seed)
    return data, False, npz_path


def preprocess_and_select(
    X_raw: pd.DataFrame,
    y: pd.Series,
    search_features: int,
    random_seed: int,
    source: str,
) -> PreparedData:
    original_features = X_raw.shape[1]
    X = X_raw.copy()

    numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    if numeric_cols:
        X[numeric_cols] = X[numeric_cols].fillna(-999)
    for col in categorical_cols:
        values = X[col].astype("string").fillna("unknown")
        X[col] = pd.factorize(values, sort=True)[0].astype("int32")

    X = X.replace([np.inf, -np.inf], -999)
    X_np = X.to_numpy(dtype=np.float32, copy=True)
    y_np = y.to_numpy(dtype=np.int8, copy=True)
    names = X.columns.tolist()

    variance = VarianceThreshold(threshold=0.0)
    X_var = variance.fit_transform(X_np)
    kept_names = [name for name, keep in zip(names, variance.get_support()) if keep]
    if X_var.shape[1] == 0:
        raise ValueError("VarianceThreshold 後沒有可用特徵。")

    k = min(search_features, X_var.shape[1])
    mi = mutual_info_classif(X_var, y_np, discrete_features=False, random_state=random_seed)
    top_idx = np.argsort(mi)[::-1][:k]

    X_selected = X_var[:, top_idx].astype(np.float32, copy=False)
    selected_names = [kept_names[i] for i in top_idx]

    return PreparedData(
        X=X_selected,
        y=y_np,
        feature_names=selected_names,
        selected_from_original=selected_names,
        source=source,
        rows=X_selected.shape[0],
        original_features=original_features,
    )


def _cache_key(path: Path, proxy_rows: int, search_features: int, random_seed: int) -> str:
    stat = path.stat()
    payload = {
        "version": CACHE_VERSION,
        "path": str(path.resolve()).lower(),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
        "proxy_rows": proxy_rows,
        "search_features": search_features,
        "random_seed": random_seed,
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]


def _save_prepared_cache(
    data: PreparedData,
    npz_path: Path,
    meta_path: Path,
    raw_path: Path,
    proxy_rows: int,
    search_features: int,
    random_seed: int,
) -> None:
    np.savez(npz_path, X=data.X, y=data.y)
    metadata = {
        "cache_version": CACHE_VERSION,
        "raw_path": str(raw_path),
        "proxy_rows": proxy_rows,
        "search_features": search_features,
        "random_seed": random_seed,
        "feature_names": data.feature_names,
        "selected_from_original": data.selected_from_original,
        "source": data.source,
        "rows": data.rows,
        "original_features": data.original_features,
    }
    meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")


def _load_prepared_cache(npz_path: Path, meta_path: Path) -> PreparedData:
    metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    with np.load(npz_path, allow_pickle=False) as arrays:
        X = arrays["X"]
        y = arrays["y"]
    return PreparedData(
        X=X,
        y=y,
        feature_names=list(metadata["feature_names"]),
        selected_from_original=list(metadata["selected_from_original"]),
        source=str(metadata["source"]),
        rows=int(metadata["rows"]),
        original_features=int(metadata["original_features"]),
    )


In [ ]:
FULL_EVAL_CACHE_VERSION = "full-eval-selected-features-v1"


def prepare_full_evaluation_data_with_cache(
    path: Path,
    feature_names: list[str],
    cache_dir: Path,
) -> tuple[PreparedData, bool, Path]:
    """Prepare the full train_transaction.csv for final 5-fold CV on the same Top-150 search columns."""
    if not path.exists():
        raise FileNotFoundError(
            f"找不到資料檔：{path}\n"
            "請先把 Kaggle IEEE-CIS 的 train_transaction.csv 放到 data/raw/。"
        )
    cache_dir.mkdir(parents=True, exist_ok=True)
    key = _full_eval_cache_key(path, feature_names)
    npz_path = cache_dir / f"{key}.npz"
    meta_path = cache_dir / f"{key}.json"
    if npz_path.exists() and meta_path.exists():
        return _load_prepared_cache(npz_path, meta_path), True, npz_path

    usecols = ["isFraud", *feature_names]
    df = pd.read_csv(path, usecols=usecols, low_memory=False)
    if "isFraud" not in df.columns:
        raise ValueError("train_transaction.csv 中找不到 isFraud 欄位。")
    missing = [name for name in feature_names if name not in df.columns]
    if missing:
        raise ValueError(f"完整訓練集缺少這些搜尋特徵：{missing[:10]}")

    y = df.pop("isFraud").astype("int8")
    X = df[feature_names].copy()
    numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]
    if numeric_cols:
        X[numeric_cols] = X[numeric_cols].fillna(-999)
    for col in categorical_cols:
        values = X[col].astype("string").fillna("unknown")
        X[col] = pd.factorize(values, sort=True)[0].astype("int32")
    X = X.replace([np.inf, -np.inf], -999)
    data = PreparedData(
        X=X.to_numpy(dtype=np.float32, copy=True),
        y=y.to_numpy(dtype=np.int8, copy=True),
        feature_names=list(feature_names),
        selected_from_original=list(feature_names),
        source=str(path),
        rows=len(y),
        original_features=len(feature_names),
    )
    _save_full_eval_cache(data, npz_path, meta_path, path, feature_names)
    return data, False, npz_path


def _full_eval_cache_key(path: Path, feature_names: list[str]) -> str:
    stat = path.stat()
    payload = {
        "version": FULL_EVAL_CACHE_VERSION,
        "path": str(path.resolve()).lower(),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
        "feature_names": feature_names,
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return "full_" + hashlib.sha256(raw).hexdigest()[:16]


def _save_full_eval_cache(
    data: PreparedData,
    npz_path: Path,
    meta_path: Path,
    raw_path: Path,
    feature_names: list[str],
) -> None:
    np.savez(npz_path, X=data.X, y=data.y)
    metadata = {
        "cache_version": FULL_EVAL_CACHE_VERSION,
        "raw_path": str(raw_path),
        "feature_names": feature_names,
        "selected_from_original": data.selected_from_original,
        "source": data.source,
        "rows": data.rows,
        "original_features": data.original_features,
    }
    meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
from dataclasses import dataclass
from time import perf_counter

import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split


@dataclass(frozen=True)
class FitnessResult:
    fitness: float
    auc: float
    selected_count: int
    selected_ratio: float
    seconds: float
    cached: bool


class FitnessEvaluator:
    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        penalty_lambda: float,
        cv_folds: int,
        test_size: float,
        n_estimators: int,
        max_depth: int,
        learning_rate: float,
        device: str,
        random_seed: int,
        n_jobs: int = 8,
        use_cache: bool = True,
    ) -> None:
        self.X = X
        self.y = y
        self.penalty_lambda = penalty_lambda
        self.cv_folds = cv_folds
        self.test_size = test_size
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.device = device
        self.random_seed = random_seed
        self.n_jobs = n_jobs
        self.use_cache = use_cache
        self.cache: dict[str, FitnessResult] = {}
        self.model_fits = 0

    @property
    def total_features(self) -> int:
        return self.X.shape[1]

    def evaluate(self, mask: np.ndarray) -> FitnessResult:
        mask = normalize_mask(mask)
        key = pack_mask(mask)
        if self.use_cache and key in self.cache:
            cached = self.cache[key]
            return FitnessResult(
                fitness=cached.fitness,
                auc=cached.auc,
                selected_count=cached.selected_count,
                selected_ratio=cached.selected_ratio,
                seconds=0.0,
                cached=True,
            )

        selected_count = int(mask.sum())
        selected_ratio = selected_count / self.total_features
        if selected_count == 0:
            result = FitnessResult(
                fitness=0.5,
                auc=0.5,
                selected_count=0,
                selected_ratio=0.0,
                seconds=0.0,
                cached=False,
            )
            if self.use_cache:
                self.cache[key] = result
            return result

        t0 = perf_counter()
        X_sub = self.X[:, mask]
        auc = self._score_auc(X_sub)
        seconds = perf_counter() - t0
        fitness = auc - self.penalty_lambda * selected_ratio
        result = FitnessResult(
            fitness=float(fitness),
            auc=float(auc),
            selected_count=selected_count,
            selected_ratio=float(selected_ratio),
            seconds=seconds,
            cached=False,
        )
        if self.use_cache:
            self.cache[key] = result
        return result

    def _score_auc(self, X_sub: np.ndarray) -> float:
        if self.cv_folds <= 1:
            X_train, X_valid, y_train, y_valid = train_test_split(
                X_sub,
                self.y,
                test_size=self.test_size,
                stratify=self.y,
                random_state=self.random_seed,
            )
            return self._fit_predict_auc(X_train, y_train, X_valid, y_valid)

        splitter = StratifiedKFold(
            n_splits=self.cv_folds,
            shuffle=True,
            random_state=self.random_seed,
        )
        aucs: list[float] = []
        for train_idx, valid_idx in splitter.split(X_sub, self.y):
            aucs.append(
                self._fit_predict_auc(
                    X_sub[train_idx],
                    self.y[train_idx],
                    X_sub[valid_idx],
                    self.y[valid_idx],
                )
            )
        return float(np.mean(aucs))

    def _fit_predict_auc(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray,
    ) -> float:
        params = {
            "n_estimators": self.n_estimators,
            "max_depth": self.max_depth,
            "learning_rate": self.learning_rate,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "eval_metric": "auc",
            "random_state": self.random_seed,
            "n_jobs": self.n_jobs,
            "tree_method": "hist",
        }
        if self.device == "cuda":
            params["device"] = "cuda"

        pos = max(1, int(np.sum(y_train == 1)))
        neg = max(1, int(np.sum(y_train == 0)))
        params["scale_pos_weight"] = neg / pos

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train, verbose=False)
        self.model_fits += 1
        pred = model.predict_proba(X_valid)[:, 1]
        return float(roc_auc_score(y_valid, pred))


def normalize_mask(mask: np.ndarray) -> np.ndarray:
    clean = np.asarray(mask, dtype=bool).copy()
    if clean.ndim != 1:
        raise ValueError("Feature mask must be one-dimensional.")
    return clean


def pack_mask(mask: np.ndarray) -> str:
    return "".join("1" if value else "0" for value in mask.astype(bool))


In [ ]:
from dataclasses import dataclass
from math import exp, log
from time import perf_counter
from typing import Callable

import numpy as np



@dataclass
class SearchResult:
    method: str
    penalty_lambda: float
    mask: np.ndarray
    result: FitnessResult
    seconds: float
    history: list[dict[str, float | int | str]]


def ensure_non_empty(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    mask = normalize_mask(mask)
    if not mask.any():
        mask[int(rng.integers(0, len(mask)))] = True
    return mask


def random_mask(n_features: int, rng: np.random.Generator, p: float = 0.5) -> np.ndarray:
    return ensure_non_empty(rng.random(n_features) < p, rng)


def run_random_search(
    evaluator: FitnessEvaluator,
    trials: int,
    rng: np.random.Generator,
    penalty_lambda: float,
    progress_callback: Callable[[str], None] | None = None,
) -> SearchResult:
    t0 = perf_counter()
    best_mask = random_mask(evaluator.total_features, rng)
    best_result = evaluator.evaluate(best_mask)
    history: list[dict[str, float | int | str]] = []

    for step in range(1, trials + 1):
        p = float(rng.uniform(0.15, 0.65))
        mask = random_mask(evaluator.total_features, rng, p=p)
        result = evaluator.evaluate(mask)
        if result.fitness > best_result.fitness:
            best_mask = mask
            best_result = result
        history.append(
            _history_row(
                method="RandomSearch",
                penalty_lambda=penalty_lambda,
                step=step,
                best=best_result,
                avg_fitness=result.fitness,
            )
        )
        if progress_callback and _should_report(step, trials):
            progress_callback(
                f"RandomSearch {step}/{trials}: "
                f"best_auc={best_result.auc:.4f}, features={best_result.selected_count}"
            )

    return SearchResult(
        method="RandomSearch",
        penalty_lambda=penalty_lambda,
        mask=best_mask,
        result=best_result,
        seconds=perf_counter() - t0,
        history=history,
    )


def run_ga(
    evaluator: FitnessEvaluator,
    population_size: int,
    generations: int,
    elites: int,
    tournament_k: int,
    crossover_rate: float,
    mutation_rate: float,
    rng: np.random.Generator,
    penalty_lambda: float,
    progress_callback: Callable[[str], None] | None = None,
) -> SearchResult:
    t0 = perf_counter()
    n_features = evaluator.total_features
    population = np.array([random_mask(n_features, rng) for _ in range(population_size)])
    scored = [evaluator.evaluate(ind) for ind in population]
    history: list[dict[str, float | int | str]] = []

    for generation in range(1, generations + 1):
        order = np.argsort([res.fitness for res in scored])[::-1]
        population = population[order]
        scored = [scored[i] for i in order]
        next_population = [population[i].copy() for i in range(min(elites, population_size))]

        while len(next_population) < population_size:
            parent_a = _tournament(population, scored, tournament_k, rng)
            parent_b = _tournament(population, scored, tournament_k, rng)
            child_a, child_b = _crossover(parent_a, parent_b, crossover_rate, rng)
            child_a = _mutate(child_a, mutation_rate, rng)
            child_b = _mutate(child_b, mutation_rate, rng)
            next_population.append(ensure_non_empty(child_a, rng))
            if len(next_population) < population_size:
                next_population.append(ensure_non_empty(child_b, rng))

        population = np.array(next_population)
        scored = [evaluator.evaluate(ind) for ind in population]
        best = max(scored, key=lambda res: res.fitness)
        avg = float(np.mean([res.fitness for res in scored]))
        history.append(
            _history_row(
                method="GA",
                penalty_lambda=penalty_lambda,
                step=generation,
                best=best,
                avg_fitness=avg,
            )
        )
        if progress_callback and _should_report(generation, generations):
            progress_callback(
                f"GA generation {generation}/{generations}: "
                f"best_auc={best.auc:.4f}, best_fitness={best.fitness:.4f}, "
                f"features={best.selected_count}"
            )

    best_idx = int(np.argmax([res.fitness for res in scored]))
    return SearchResult(
        method="GA",
        penalty_lambda=penalty_lambda,
        mask=population[best_idx].copy(),
        result=scored[best_idx],
        seconds=perf_counter() - t0,
        history=history,
    )


def run_sa(
    evaluator: FitnessEvaluator,
    iterations: int,
    alpha: float,
    t_min: float,
    calibration_moves: int,
    log_interval: int,
    rng: np.random.Generator,
    penalty_lambda: float,
    progress_callback: Callable[[str], None] | None = None,
) -> SearchResult:
    t0 = perf_counter()
    current = random_mask(evaluator.total_features, rng)
    current_result = evaluator.evaluate(current)
    best = current.copy()
    best_result = current_result
    temperature = _calibrate_temperature(evaluator, current, current_result, calibration_moves, rng)
    history: list[dict[str, float | int | str]] = []

    for iteration in range(1, iterations + 1):
        candidate = _neighbor(current, rng)
        candidate_result = evaluator.evaluate(candidate)
        delta = candidate_result.fitness - current_result.fitness
        if delta >= 0 or rng.random() < exp(delta / max(temperature, 1e-12)):
            current = candidate
            current_result = candidate_result
        if current_result.fitness > best_result.fitness:
            best = current.copy()
            best_result = current_result

        temperature = max(t_min, temperature * alpha)
        if iteration == 1 or iteration % log_interval == 0 or iteration == iterations:
            history.append(
                _history_row(
                    method="SA",
                    penalty_lambda=penalty_lambda,
                    step=iteration,
                    best=best_result,
                    avg_fitness=current_result.fitness,
                    temperature=temperature,
                )
            )
            if progress_callback:
                progress_callback(
                    f"SA iteration {iteration}/{iterations}: "
                    f"best_auc={best_result.auc:.4f}, best_fitness={best_result.fitness:.4f}, "
                    f"features={best_result.selected_count}, T={temperature:.5f}"
                )

    return SearchResult(
        method="SA",
        penalty_lambda=penalty_lambda,
        mask=best,
        result=best_result,
        seconds=perf_counter() - t0,
        history=history,
    )


def _tournament(
    population: np.ndarray,
    scored: list[FitnessResult],
    k: int,
    rng: np.random.Generator,
) -> np.ndarray:
    idx = rng.choice(len(population), size=min(k, len(population)), replace=False)
    winner = max(idx, key=lambda i: scored[int(i)].fitness)
    return population[int(winner)].copy()


def _crossover(
    parent_a: np.ndarray,
    parent_b: np.ndarray,
    crossover_rate: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    if len(parent_a) < 2 or rng.random() >= crossover_rate:
        return parent_a.copy(), parent_b.copy()
    point = int(rng.integers(1, len(parent_a)))
    child_a = np.concatenate([parent_a[:point], parent_b[point:]])
    child_b = np.concatenate([parent_b[:point], parent_a[point:]])
    return child_a, child_b


def _mutate(mask: np.ndarray, mutation_rate: float, rng: np.random.Generator) -> np.ndarray:
    child = mask.copy()
    flips = rng.random(len(child)) < mutation_rate
    child[flips] = ~child[flips]
    return child


def _neighbor(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    candidate = mask.copy()
    flips = int(rng.integers(1, 4))
    idx = rng.choice(len(candidate), size=min(flips, len(candidate)), replace=False)
    candidate[idx] = ~candidate[idx]
    return ensure_non_empty(candidate, rng)


def _calibrate_temperature(
    evaluator: FitnessEvaluator,
    current: np.ndarray,
    current_result: FitnessResult,
    calibration_moves: int,
    rng: np.random.Generator,
) -> float:
    bad_deltas: list[float] = []
    for _ in range(calibration_moves):
        candidate = _neighbor(current, rng)
        result = evaluator.evaluate(candidate)
        delta = result.fitness - current_result.fitness
        if delta < 0:
            bad_deltas.append(abs(delta))
    if not bad_deltas:
        return 0.02
    return max(float(np.mean(bad_deltas)) / -log(0.8), 1e-4)


def _history_row(
    method: str,
    penalty_lambda: float,
    step: int,
    best: FitnessResult,
    avg_fitness: float,
    temperature: float | None = None,
) -> dict[str, float | int | str]:
    row: dict[str, float | int | str] = {
        "method": method,
        "lambda": penalty_lambda,
        "step": step,
        "best_fitness": best.fitness,
        "best_auc": best.auc,
        "best_selected_count": best.selected_count,
        "avg_or_current_fitness": avg_fitness,
    }
    if temperature is not None:
        row["temperature"] = temperature
    return row


def _should_report(step: int, total: int) -> bool:
    interval = max(1, total // 10)
    return step == 1 or step % interval == 0 or step == total


In [ ]:
import json
from dataclasses import asdict
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd



def make_run_dir(results_dir: Path) -> Path:
    from datetime import datetime

    run_dir = results_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    (run_dir / "plots").mkdir(parents=True, exist_ok=False)
    return run_dir


def write_outputs(
    run_dir: Path,
    config: ExperimentConfig,
    data: PreparedData,
    metrics: list[dict[str, object]],
    selected: dict[str, list[str]],
    convergence: list[dict[str, object]],
    extra_metadata: dict[str, object] | None = None,
) -> None:
    metrics_df = pd.DataFrame(metrics)
    convergence_df = pd.DataFrame(convergence)

    metrics_df.to_csv(run_dir / "metrics.csv", index=False, encoding="utf-8-sig")
    convergence_df.to_csv(run_dir / "convergence.csv", index=False, encoding="utf-8-sig")
    (run_dir / "selected_features.json").write_text(
        json.dumps(selected, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    metadata = {
        "config": asdict(config),
        "data": {
            "source": data.source,
            "rows": data.rows,
            "original_features": data.original_features,
            "search_features": len(data.feature_names),
        },
    }
    if extra_metadata:
        metadata.update(extra_metadata)
    (run_dir / "run_metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    _plot_convergence(convergence_df, run_dir / "plots" / "convergence.png")
    _plot_pareto(metrics_df, run_dir / "plots" / "pareto_auc_vs_features.png")
    _plot_lambda(metrics_df, run_dir / "plots" / "lambda_sensitivity.png")
    _plot_feature_selection_by_mi_rank(
        data.feature_names,
        selected,
        run_dir / "plots" / "feature_importance_selection.png",
    )
    _write_feature_overlap(run_dir, metrics_df, selected)
    _write_summary(run_dir, config, data, metrics_df)


def selected_feature_names(feature_names: list[str], mask: Iterable[bool]) -> list[str]:
    return [name for name, keep in zip(feature_names, mask) if keep]


def _plot_convergence(df: pd.DataFrame, path: Path) -> None:
    if df.empty:
        return

    main = df[df["method"].isin(["GA", "SA"])].copy()
    if main.empty:
        return

    max_steps = main.groupby(["lambda", "method"])["step"].transform("max")
    main["progress_pct"] = main["step"] / max_steps * 100.0
    lambdas = sorted(main["lambda"].unique())

    fig, axes = plt.subplots(2, 2, figsize=(18, 11), sharex=True)
    axes = axes.flatten()
    for ax, lam in zip(axes, lambdas):
        group = main[main["lambda"] == lam]
        ga = group[group["method"] == "GA"].sort_values("progress_pct")
        sa = group[group["method"] == "SA"].sort_values("progress_pct")

        if not ga.empty:
            ax.plot(
                ga["progress_pct"],
                ga["best_fitness"],
                color="#1f77b4",
                linewidth=2.2,
                label="GA best fitness",
            )
            ax.plot(
                ga["progress_pct"],
                ga["avg_or_current_fitness"],
                color="#ff7f0e",
                linewidth=1.9,
                linestyle="--",
                label="GA average fitness",
            )
        if not sa.empty:
            ax.plot(
                sa["progress_pct"],
                sa["best_fitness"],
                color="#2ca02c",
                linewidth=2.2,
                label="SA best fitness",
            )

        ax.set_title(f"λ = {lam:g}", fontsize=15, pad=10)
        ax.set_xlim(0, 100)
        ax.grid(alpha=0.25)
        ax.set_xlabel("搜尋進度（%）", fontsize=12)
        ax.set_ylabel("Fitness", fontsize=12)

    for ax in axes[len(lambdas):]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=12, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle("圖一：GA/SA 收斂曲線比較", fontsize=18, y=0.99)
    fig.text(
        0.012,
        0.015,
        "說明：GA 為 60 generations、SA 為 5000 iterations；為了讓兩者在同一張圖中清楚比較，x 軸以各方法搜尋進度百分比呈現。",
        fontsize=11,
        color="#374151",
    )
    fig.tight_layout(rect=[0, 0.04, 1, 0.93])
    plt.savefig(path, dpi=220)
    plt.close()


def _plot_pareto(df: pd.DataFrame, path: Path) -> None:
    if df.empty:
        return
    auc_col = "final_cv_auc" if "final_cv_auc" in df.columns else "auc"
    plt.figure(figsize=(9, 6))
    for method, group in df.groupby("method"):
        plt.scatter(group["selected_count"], group[auc_col], label=method, s=60)
    plt.xlabel("選取特徵數")
    plt.ylabel("AUC")
    plt.title("圖三：Pareto Front - AUC vs 特徵數")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def _plot_lambda(df: pd.DataFrame, path: Path) -> None:
    main = df[df["method"].isin(["GA", "SA", "RandomSearch"])]
    if main.empty:
        return
    auc_col = "final_cv_auc" if "final_cv_auc" in main.columns else "auc"
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    for method, group in main.groupby("method"):
        group = group.sort_values("lambda")
        ax1.plot(group["lambda"], group[auc_col], marker="o", label=f"{method} AUC")
        ax2.plot(
            group["lambda"],
            group["selected_count"],
            marker="x",
            linestyle="--",
            label=f"{method} features",
        )
    ax1.set_xlabel("懲罰係數 λ")
    ax1.set_ylabel("AUC")
    ax2.set_ylabel("選取特徵數")
    ax1.grid(alpha=0.25)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    plt.title("圖四：λ 敏感度分析 - AUC 與特徵數取捨")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def _plot_feature_selection_by_mi_rank(
    feature_names: list[str],
    selected: dict[str, list[str]],
    path: Path,
    top_n: int = 30,
) -> None:
    if not feature_names or not selected:
        return

    lambdas: list[str] = []
    for key in selected:
        if "::" not in key or "::GA" not in key:
            continue
        lam = key.split("::", 1)[0].replace("lambda=", "")
        lambdas.append(lam)
    preferred = [lam for lam in ["0.05", "0.1", "0.10"] if lam in lambdas]
    chosen_lambdas = preferred[:2] if preferred else sorted(set(lambdas), key=float)[:2]
    if not chosen_lambdas:
        return

    top_features = feature_names[:top_n]
    display_features = [f"{idx + 1:02d}. {name}" for idx, name in enumerate(top_features)]

    columns: list[tuple[str, set[str]]] = []
    for lam in chosen_lambdas:
        for method in ("GA", "SA"):
            values = selected.get(f"lambda={lam}::{method}")
            if values is not None:
                columns.append((f"{method}\nλ={float(lam):g}", set(values)))
    if not columns:
        return

    matrix = np.array(
        [[1 if feature in feature_set else 0 for _, feature_set in columns] for feature in top_features],
        dtype=int,
    )
    rank_score = np.arange(top_n, 0, -1)
    selected_count = matrix.sum(axis=1)

    fig, (ax_bar, ax_sel, ax_count) = plt.subplots(
        ncols=3,
        figsize=(18, 13),
        gridspec_kw={"width_ratios": [3.4, 1.35, 1.0]},
        sharey=True,
    )
    y_pos = np.arange(top_n)

    ax_bar.barh(y_pos, rank_score, color="#3b6ea8", alpha=0.92)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(display_features, fontsize=12)
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("MI 排名分數（越前面越重要）", fontsize=12)
    ax_bar.set_title("Top 30 MI 特徵排名", fontsize=15, pad=12)
    ax_bar.grid(axis="x", alpha=0.22)

    from matplotlib.colors import ListedColormap

    cmap = ListedColormap(["#f7f7d5", "#0b2a66"])
    ax_sel.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=1)
    ax_sel.set_xticks(np.arange(len(columns)))
    ax_sel.set_xticklabels([label for label, _ in columns], fontsize=12)
    ax_sel.set_yticks(y_pos)
    ax_sel.tick_params(axis="y", left=False, labelleft=False)
    ax_sel.set_title("GA / SA 選取狀態", fontsize=15, pad=12)
    ax_sel.set_xticks(np.arange(-0.5, len(columns), 1), minor=True)
    ax_sel.set_yticks(np.arange(-0.5, top_n, 1), minor=True)
    ax_sel.grid(which="minor", color="white", linestyle="-", linewidth=1.3)
    ax_sel.tick_params(which="minor", bottom=False, left=False)

    ax_count.barh(y_pos, selected_count, color="#2f9e75", alpha=0.9)
    ax_count.set_xlim(0, max(1, len(columns)))
    ax_count.set_xticks(range(0, len(columns) + 1))
    ax_count.tick_params(axis="y", left=False, labelleft=False)
    ax_count.set_xlabel("次數", fontsize=12)
    ax_count.set_title("被選取次數", fontsize=15, pad=12)
    ax_count.grid(axis="x", alpha=0.22)

    handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12, markerfacecolor="#0b2a66", markeredgecolor="#0b2a66", label="有選取"),
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12, markerfacecolor="#f7f7d5", markeredgecolor="#c8c8a0", label="未選取"),
    ]
    fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.985, 0.965), frameon=False, fontsize=12)
    fig.suptitle("圖二：MI 特徵重要性與 GA/SA 選取狀態", fontsize=18, y=0.985)
    fig.text(
        0.012,
        0.012,
        "說明：為避免 HTML 縮放後文字模糊，選取狀態以色塊呈現；深藍表示該方法有選取該特徵。",
        fontsize=11,
        color="#374151",
    )
    fig.tight_layout(rect=[0.0, 0.035, 0.98, 0.955])
    plt.savefig(path, dpi=220)
    plt.close()


def _write_feature_overlap(
    run_dir: Path,
    metrics_df: pd.DataFrame,
    selected: dict[str, list[str]],
) -> None:
    rows: list[dict[str, float | int]] = []
    for lam in sorted(metrics_df["lambda"].dropna().unique()):
        lam_text = str(lam)
        ga = set(selected.get(f"lambda={lam_text}::GA", []))
        sa = set(selected.get(f"lambda={lam_text}::SA", []))
        if not ga or not sa:
            continue
        overlap = len(ga & sa)
        union = len(ga | sa)
        rows.append(
            {
                "lambda": float(lam),
                "ga_features": len(ga),
                "sa_features": len(sa),
                "overlap": overlap,
                "jaccard_similarity": overlap / union if union else 0.0,
            }
        )
    if rows:
        pd.DataFrame(rows).to_csv(
            run_dir / "feature_overlap_ga_sa.csv",
            index=False,
            encoding="utf-8-sig",
        )


def _write_summary(
    run_dir: Path,
    config: ExperimentConfig,
    data: PreparedData,
    metrics_df: pd.DataFrame,
) -> None:
    score_col = "final_cv_fitness" if "final_cv_fitness" in metrics_df.columns else "fitness"
    auc_col = "final_cv_auc" if "final_cv_auc" in metrics_df.columns else "auc"
    best_rows = metrics_df.sort_values(["lambda", score_col], ascending=[True, False])
    lines = [
        "# GA/SA Feature Selection Run Summary",
        "",
        f"- preset: `{config.preset}`",
        f"- source: `{data.source}`",
        f"- rows: `{data.rows}`",
        f"- original features: `{data.original_features}`",
        f"- search features after preprocessing/MI: `{len(data.feature_names)}`",
        f"- inner validation folds: `{config.inner_cv_folds}` (`1` means holdout)",
        f"- final CV folds: `{config.final_cv_folds}`",
        "",
        "## Best Method Per Lambda",
        "",
    ]
    for lam, group in best_rows.groupby("lambda", sort=True):
        row = group.iloc[0]
        lines.append(
            f"- lambda `{lam}`: `{row['method']}` "
            f"final AUC=`{row[auc_col]:.4f}`, final fitness=`{row[score_col]:.4f}`, "
            f"features=`{int(row['selected_count'])}`"
        )

    lines.extend(
        [
            "",
            "## All Metrics",
            "",
            "```text",
            metrics_df.to_string(index=False),
            "```",
            "",
            "## Files",
            "",
            "- `metrics.csv`",
            "- `selected_features.json`",
            "- `convergence.csv`",
            "- `plots/convergence.png`",
            "- `plots/pareto_auc_vs_features.png`",
            "- `plots/lambda_sensitivity.png`",
            "- `plots/feature_importance_selection.png`",
            "- `feature_overlap_ga_sa.csv`",
        ]
    )
    (run_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")


In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Iterable

import pandas as pd


def make_aggregate_dir(results_dir: Path) -> Path:
    out_dir = results_dir / f"aggregate_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=False)
    return out_dir


def aggregate_runs(run_dirs: Iterable[Path], out_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[pd.DataFrame] = []
    metadata_rows: list[dict[str, object]] = []

    for run_dir in run_dirs:
        run_dir = Path(run_dir)
        metrics_path = run_dir / "metrics.csv"
        if not metrics_path.exists():
            continue

        metrics = pd.read_csv(metrics_path)
        metadata = _read_metadata(run_dir)
        config = metadata.get("config", {}) if isinstance(metadata.get("config"), dict) else {}
        data = metadata.get("data", {}) if isinstance(metadata.get("data"), dict) else {}

        metrics["run_dir"] = str(run_dir)
        metrics["preset"] = config.get("preset", "unknown")
        metrics["seed"] = config.get("random_seed", "unknown")
        metrics["mode"] = metadata.get("mode", "unknown")
        metrics["device"] = metadata.get("device", "unknown")
        metrics["search_device"] = metadata.get("search_device", metadata.get("device", "unknown"))
        metrics["final_device"] = metadata.get("final_device", metadata.get("device", "unknown"))
        metrics["rows"] = data.get("rows", None)
        metrics["search_features"] = data.get("search_features", None)
        rows.append(metrics)

        metadata_rows.append(
            {
                "run_dir": str(run_dir),
                "preset": config.get("preset", "unknown"),
                "seed": config.get("random_seed", "unknown"),
                "mode": metadata.get("mode", "unknown"),
                "device": metadata.get("device", "unknown"),
                "search_device": metadata.get("search_device", metadata.get("device", "unknown")),
                "final_device": metadata.get("final_device", metadata.get("device", "unknown")),
                "rows": data.get("rows", None),
                "search_features": data.get("search_features", None),
                "lambdas": ",".join(str(x) for x in config.get("lambdas", [])) if isinstance(config.get("lambdas", []), list) else str(config.get("lambdas", "")),
                "random_trials": config.get("random_trials", None),
                "ga_population": config.get("ga_population", None),
                "ga_generations": config.get("ga_generations", None),
                "sa_iterations": config.get("sa_iterations", None),
                "inner_cv_folds": config.get("inner_cv_folds", None),
                "final_cv_folds": config.get("final_cv_folds", None),
                "xgb_estimators_inner": config.get("xgb_estimators_inner", None),
                "xgb_estimators_final": config.get("xgb_estimators_final", None),
            }
        )

    if not rows:
        raise FileNotFoundError("No metrics.csv files found in the supplied run directories.")

    all_metrics = pd.concat(rows, ignore_index=True)
    group_cols = ["lambda", "method"]
    summary = (
        all_metrics.groupby(group_cols, dropna=False)
        .agg(
            n_runs=("run_dir", "nunique"),
            final_cv_auc_mean=("final_cv_auc", "mean"),
            final_cv_auc_std=("final_cv_auc", "std"),
            final_cv_fitness_mean=("final_cv_fitness", "mean"),
            final_cv_fitness_std=("final_cv_fitness", "std"),
            selected_count_mean=("selected_count", "mean"),
            selected_count_std=("selected_count", "std"),
            inner_auc_mean=("auc", "mean"),
            inner_auc_std=("auc", "std"),
            seconds_mean=("seconds", "mean"),
        )
        .reset_index()
        .sort_values(["lambda", "final_cv_fitness_mean"], ascending=[True, False])
    )
    summary = summary.fillna(
        {
            "final_cv_auc_std": 0.0,
            "final_cv_fitness_std": 0.0,
            "selected_count_std": 0.0,
            "inner_auc_std": 0.0,
        }
    )

    out_dir.mkdir(parents=True, exist_ok=True)
    all_metrics.to_csv(out_dir / "all_metrics.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(out_dir / "aggregate_metrics.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(metadata_rows).to_csv(out_dir / "runs.csv", index=False, encoding="utf-8-sig")
    _write_summary_md(out_dir, summary, pd.DataFrame(metadata_rows))
    return all_metrics, summary


def _read_metadata(run_dir: Path) -> dict[str, object]:
    path = run_dir / "run_metadata.json"
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def _write_summary_md(out_dir: Path, summary: pd.DataFrame, runs: pd.DataFrame) -> None:
    full_seed_count = int(summary["n_runs"].max())
    lines = [
        "# Multi-Seed Aggregate Summary",
        "",
        "## Runs",
        "",
        "```text",
        runs.to_string(index=False),
        "```",
        "",
        "## Best Method Per Lambda",
        "",
        f"Only methods present in all `{full_seed_count}` runs are considered here. "
        "Rows with fewer runs are still listed in the full metric table below.",
        "",
    ]

    for lam, group in summary.groupby("lambda", sort=True):
        comparable = group[group["n_runs"] == full_seed_count]
        if comparable.empty:
            comparable = group
        best_auc = comparable.sort_values("final_cv_auc_mean", ascending=False).iloc[0]
        best_fitness = comparable.sort_values("final_cv_fitness_mean", ascending=False).iloc[0]
        lines.append(
            f"- lambda `{lam}`: best mean AUC = `{best_auc['method']}` "
            f"({best_auc['final_cv_auc_mean']:.4f} +/- {best_auc['final_cv_auc_std']:.4f}, "
            f"features {best_auc['selected_count_mean']:.1f}); "
            f"best mean fitness = `{best_fitness['method']}` "
            f"({best_fitness['final_cv_fitness_mean']:.4f} +/- {best_fitness['final_cv_fitness_std']:.4f}, "
            f"features {best_fitness['selected_count_mean']:.1f})"
        )

    lines.extend(
        [
            "",
            "## Aggregate Metrics",
            "",
            "```text",
            summary.to_string(index=False),
            "```",
            "",
            "## Files",
            "",
            "- `all_metrics.csv`: every metric row from every run",
            "- `aggregate_metrics.csv`: mean/std by lambda and method",
            "- `runs.csv`: run directory and metadata",
        ]
    )
    (out_dir / "aggregate_summary.md").write_text("\n".join(lines), encoding="utf-8")


## Notebook 內建執行與中文報告函式

下面這格取代原本的 `run_experiment.py`、`run_multi_seed.py` 與中文彙整腳本。


In [ ]:
def resolve_device(device: str) -> str:
    """Return cpu/cuda. Use cuda for auto only when an NVIDIA runtime is visible."""
    if device in {"cpu", "cuda"}:
        return device
    return "cuda" if shutil.which("nvidia-smi") else "cpu"


def run_single_experiment(
    mode: str = "real",
    preset: str = "word_strict",
    data_path: Path = DATA_PATH,
    results_dir: Path = RESULTS_DIR,
    cache_dir: Path = CACHE_DIR,
    no_cache: bool = False,
    device: str = "auto",
    final_device: str | None = None,
    proxy_rows: int | None = None,
    search_features: int | None = None,
    seed: int | None = None,
    final_eval_full_training_set: bool = True,
) -> Path:
    """Run one GA/SA feature-selection experiment entirely inside this notebook."""
    actual_device = resolve_device(device)
    actual_final_device = resolve_device(final_device or device)
    config = get_notebook_config(
        preset,
        proxy_rows=proxy_rows,
        search_features=search_features,
        random_seed=seed,
    )
    run_dir = make_run_dir(results_dir)

    print(
        f"[1/6] 載入搜尋資料：mode={mode}, preset={config.preset}, "
        f"search_device={actual_device}, final_device={actual_final_device}",
        flush=True,
    )
    if mode == "synthetic":
        X_raw, y = load_synthetic(config.proxy_rows, config.random_seed)
        source = "synthetic"
        print("[2/6] 前處理並用 mutual information 選搜尋特徵", flush=True)
        data = preprocess_and_select(
            X_raw=X_raw,
            y=y,
            search_features=config.search_features,
            random_seed=config.random_seed,
            source=source,
        )
    elif no_cache:
        X_raw, y = load_ieee_transaction(data_path, config.proxy_rows, config.random_seed)
        source = str(data_path)
        print("[2/6] 前處理並用 mutual information 選搜尋特徵（未使用快取）", flush=True)
        data = preprocess_and_select(
            X_raw=X_raw,
            y=y,
            search_features=config.search_features,
            random_seed=config.random_seed,
            source=source,
        )
    else:
        print("[2/6] 檢查 proxy 前處理快取", flush=True)
        data_start = perf_counter()
        data, cache_hit, cache_path = prepare_ieee_transaction_with_cache(
            path=data_path,
            proxy_rows=config.proxy_rows,
            search_features=config.search_features,
            random_seed=config.random_seed,
            cache_dir=cache_dir,
        )
        state = "hit" if cache_hit else "miss/rebuilt"
        print(f"      proxy cache {state}: {cache_path} ({perf_counter() - data_start:.1f}s)", flush=True)

    print(
        f"      proxy_rows={data.rows}, original_features={data.original_features}, "
        f"search_features={len(data.feature_names)}",
        flush=True,
    )

    metrics: list[dict[str, object]] = []
    selected: dict[str, list[str]] = {}
    convergence: list[dict[str, object]] = []
    total_start = perf_counter()

    for lam_idx, penalty_lambda in enumerate(config.lambdas, start=1):
        print(f"[3/6] Lambda {lam_idx}/{len(config.lambdas)}: {penalty_lambda}", flush=True)
        rng = np.random.default_rng(config.random_seed + int(penalty_lambda * 10_000))
        evaluator = FitnessEvaluator(
            X=data.X,
            y=data.y,
            penalty_lambda=penalty_lambda,
            cv_folds=config.inner_cv_folds,
            test_size=config.test_size,
            n_estimators=config.xgb_estimators_inner,
            max_depth=config.xgb_max_depth,
            learning_rate=config.xgb_learning_rate,
            device=actual_device,
            random_seed=config.random_seed,
        )

        all_mask = np.ones(data.X.shape[1], dtype=bool)
        all_result = evaluator.evaluate(all_mask)
        _add_metric(metrics, selected, data.feature_names, "AllSearchFeatures", penalty_lambda, all_mask, all_result, all_result.seconds, evaluator.model_fits)

        random_result = run_random_search(evaluator, config.random_trials, rng, penalty_lambda, progress_callback=_progress)
        _record_search(random_result, metrics, selected, convergence, data.feature_names, evaluator.model_fits)
        print(f"      RandomSearch: auc={random_result.result.auc:.4f}, features={random_result.result.selected_count}, seconds={random_result.seconds:.1f}", flush=True)

        mutation_rate = config.ga_mutation_rate or (1.0 / data.X.shape[1])
        ga_result = run_ga(
            evaluator=evaluator,
            population_size=config.ga_population,
            generations=config.ga_generations,
            elites=config.ga_elites,
            tournament_k=config.ga_tournament_k,
            crossover_rate=config.ga_crossover_rate,
            mutation_rate=mutation_rate,
            rng=rng,
            penalty_lambda=penalty_lambda,
            progress_callback=_progress,
        )
        _record_search(ga_result, metrics, selected, convergence, data.feature_names, evaluator.model_fits)
        print(f"      GA: auc={ga_result.result.auc:.4f}, features={ga_result.result.selected_count}, seconds={ga_result.seconds:.1f}", flush=True)

        sa_result = run_sa(
            evaluator=evaluator,
            iterations=config.sa_iterations,
            alpha=config.sa_alpha,
            t_min=config.sa_t_min,
            calibration_moves=config.sa_calibration_moves,
            log_interval=config.sa_log_interval,
            rng=rng,
            penalty_lambda=penalty_lambda,
            progress_callback=_progress,
        )
        _record_search(sa_result, metrics, selected, convergence, data.feature_names, evaluator.model_fits)
        print(f"      SA: auc={sa_result.result.auc:.4f}, features={sa_result.result.selected_count}, seconds={sa_result.seconds:.1f}", flush=True)

        topk_counts = sorted({ga_result.result.selected_count, sa_result.result.selected_count})
        for k in topk_counts:
            if k <= 0:
                continue
            topk_mask = np.zeros(data.X.shape[1], dtype=bool)
            topk_mask[: min(k, data.X.shape[1])] = True
            topk_result = evaluator.evaluate(topk_mask)
            _add_metric(metrics, selected, data.feature_names, f"TopK_MI_{k}", penalty_lambda, topk_mask, topk_result, topk_result.seconds, evaluator.model_fits)

        _write_partial_outputs(run_dir, metrics, selected, convergence)
        print(f"      已寫出中途進度：{run_dir / 'partial_metrics.csv'}", flush=True)

    final_data = data
    final_metadata = {
        "final_evaluation_source": "proxy_dataset",
        "final_rows": data.rows,
        "final_features": len(data.feature_names),
    }
    if mode == "real" and final_eval_full_training_set:
        print("[4/6] 準備完整 train_transaction.csv 做 final 5-fold CV", flush=True)
        full_start = perf_counter()
        final_data, full_cache_hit, full_cache_path = prepare_full_evaluation_data_with_cache(
            path=data_path,
            feature_names=data.feature_names,
            cache_dir=cache_dir,
        )
        state = "hit" if full_cache_hit else "miss/rebuilt"
        print(
            f"      full-eval cache {state}: {full_cache_path} ({perf_counter() - full_start:.1f}s)",
            flush=True,
        )
        print(f"      final_rows={final_data.rows}, final_features={len(final_data.feature_names)}", flush=True)
        final_metadata = {
            "final_evaluation_source": "full_train_transaction",
            "final_rows": final_data.rows,
            "final_features": len(final_data.feature_names),
            "final_cache_path": str(full_cache_path),
        }

    print("[5/6] 使用 final 5-fold CV 重新評估所有候選解", flush=True)
    _final_cv_reevaluate(metrics, final_data, config, actual_final_device)

    print("[6/6] 寫出 CSV、JSON、圖表與摘要", flush=True)
    write_outputs(
        run_dir,
        config,
        data,
        metrics,
        selected,
        convergence,
        extra_metadata={
            "mode": mode,
            "device": actual_device,
            "search_device": actual_device,
            "final_device": actual_final_device,
            "data_path": str(data_path),
            "cache_dir": str(cache_dir),
            "no_cache": bool(no_cache),
            **final_metadata,
        },
    )
    print(f"完成，耗時 {perf_counter() - total_start:.1f}s", flush=True)
    print(f"結果：{run_dir}", flush=True)
    return run_dir


def run_multi_seed_experiment(
    seeds: list[int] = [7, 42, 2026],
    mode: str = "real",
    preset: str = "word_strict",
    data_path: Path = DATA_PATH,
    results_dir: Path = RESULTS_DIR,
    cache_dir: Path = CACHE_DIR,
    device: str = "auto",
    final_device: str | None = None,
    search_features: int | None = 150,
    proxy_rows: int | None = 20_000,
    no_cache: bool = False,
    final_eval_full_training_set: bool = True,
) -> Path:
    """Run the full Word-report experiment across multiple seeds, then aggregate results."""
    if not seeds:
        raise ValueError("seeds 至少要有一個整數。")
    completed: list[Path] = []
    for index, seed in enumerate(seeds, start=1):
        print(f"=== Seed {index}/{len(seeds)}: {seed} ===", flush=True)
        run_dir = run_single_experiment(
            mode=mode,
            preset=preset,
            data_path=data_path,
            results_dir=results_dir,
            cache_dir=cache_dir,
            no_cache=no_cache,
            device=device,
            final_device=final_device,
            proxy_rows=proxy_rows,
            search_features=search_features,
            seed=seed,
            final_eval_full_training_set=final_eval_full_training_set,
        )
        completed.append(run_dir)

    print("=== 彙整 multi-seed 結果 ===", flush=True)
    out_dir = make_aggregate_dir(results_dir)
    aggregate_runs(completed, out_dir)
    write_aggregate_feature_overlap(completed, out_dir)
    representative_run = completed[-1]
    copy_figures_for_report(representative_run, out_dir)
    write_chinese_aggregate_summary(out_dir)
    write_chinese_html_summary(out_dir, representative_run=representative_run)
    print("完成的 run：", flush=True)
    for run_dir in completed:
        print("  ", run_dir, flush=True)
    print("彙整輸出：", out_dir, flush=True)
    print("中文 HTML：", out_dir / "GA_SA_report_summary_zh_TW.html", flush=True)
    return out_dir


def _write_partial_outputs(run_dir: Path, metrics: list[dict[str, object]], selected: dict[str, list[str]], convergence: list[dict[str, object]]) -> None:
    pd.DataFrame(metrics).to_csv(run_dir / "partial_metrics.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(convergence).to_csv(run_dir / "partial_convergence.csv", index=False, encoding="utf-8-sig")
    (run_dir / "partial_selected_features.json").write_text(
        json.dumps(selected, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def _record_search(search_result, metrics, selected, convergence, feature_names, model_fits) -> None:
    _add_metric(metrics, selected, feature_names, search_result.method, search_result.penalty_lambda, search_result.mask, search_result.result, search_result.seconds, model_fits)
    convergence.extend(search_result.history)


def _progress(message: str) -> None:
    print(f"      {message}", flush=True)


def _add_metric(metrics, selected, feature_names, method, penalty_lambda, mask, result, seconds, model_fits) -> None:
    key = f"lambda={penalty_lambda}::{method}"
    selected[key] = selected_feature_names(feature_names, mask)
    metrics.append({
        "method": method,
        "lambda": penalty_lambda,
        "mask_bits": "".join("1" if value else "0" for value in mask.astype(bool)),
        "fitness": result.fitness,
        "auc": result.auc,
        "selected_count": result.selected_count,
        "selected_ratio": result.selected_ratio,
        "seconds": seconds,
        "model_fits_so_far": model_fits,
    })


def _final_cv_reevaluate(metrics: list[dict[str, object]], data, config, device: str) -> None:
    evaluators: dict[float, FitnessEvaluator] = {}
    total = len(metrics)
    for idx, row in enumerate(metrics, start=1):
        penalty_lambda = float(row["lambda"])
        if penalty_lambda not in evaluators:
            evaluators[penalty_lambda] = FitnessEvaluator(
                X=data.X,
                y=data.y,
                penalty_lambda=penalty_lambda,
                cv_folds=config.final_cv_folds,
                test_size=config.test_size,
                n_estimators=config.xgb_estimators_final,
                max_depth=config.xgb_max_depth,
                learning_rate=config.xgb_learning_rate,
                device=device,
                random_seed=config.random_seed + 7_777,
            )
        mask = np.array([char == "1" for char in str(row["mask_bits"])], dtype=bool)
        print(f"      Final CV {idx}/{total}: {row['method']} lambda={penalty_lambda}", flush=True)
        result = evaluators[penalty_lambda].evaluate(mask)
        row["final_cv_auc"] = result.auc
        row["final_cv_fitness"] = result.fitness
        row["final_cv_seconds"] = result.seconds
        row["final_cv_folds"] = config.final_cv_folds


def write_aggregate_feature_overlap(run_dirs: Iterable[Path], out_dir: Path) -> None:
    rows = []
    for run_dir in run_dirs:
        run_dir = Path(run_dir)
        meta_path = run_dir / "run_metadata.json"
        selected_path = run_dir / "selected_features.json"
        if not selected_path.exists():
            continue
        metadata = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
        seed = metadata.get("config", {}).get("random_seed", "unknown") if isinstance(metadata.get("config"), dict) else "unknown"
        selected = json.loads(selected_path.read_text(encoding="utf-8"))
        lambdas = sorted({key.split("::", 1)[0].replace("lambda=", "") for key in selected if "::" in key}, key=float)
        for lam in lambdas:
            ga = set(selected.get(f"lambda={lam}::GA", []))
            sa = set(selected.get(f"lambda={lam}::SA", []))
            if ga and sa:
                union = ga | sa
                rows.append({
                    "run_dir": str(run_dir),
                    "seed": seed,
                    "lambda": float(lam),
                    "ga_features": len(ga),
                    "sa_features": len(sa),
                    "overlap": len(ga & sa),
                    "jaccard_similarity": len(ga & sa) / len(union) if union else 0.0,
                })
    if not rows:
        return
    detail = pd.DataFrame(rows)
    detail.to_csv(out_dir / "feature_overlap_all_runs.csv", index=False, encoding="utf-8-sig")
    summary = detail.groupby("lambda").agg(
        n_runs=("run_dir", "nunique"),
        ga_features_mean=("ga_features", "mean"),
        sa_features_mean=("sa_features", "mean"),
        overlap_mean=("overlap", "mean"),
        jaccard_mean=("jaccard_similarity", "mean"),
        jaccard_std=("jaccard_similarity", "std"),
    ).reset_index().fillna({"jaccard_std": 0.0})
    summary.to_csv(out_dir / "feature_overlap_summary.csv", index=False, encoding="utf-8-sig")


def copy_figures_for_report(representative_run: Path, out_dir: Path) -> None:
    figures = {
        "convergence.png": "01_convergence.png",
        "feature_importance_selection.png": "02_feature_importance_selection.png",
        "pareto_auc_vs_features.png": "03_pareto_auc_vs_features.png",
        "lambda_sensitivity.png": "04_lambda_sensitivity.png",
    }
    fig_dir = out_dir / "figures_for_report"
    fig_dir.mkdir(parents=True, exist_ok=True)
    for src_name, dst_name in figures.items():
        src = representative_run / "plots" / src_name
        if src.exists():
            shutil.copyfile(src, fig_dir / dst_name)


def _read_runs_table(out_dir: Path) -> pd.DataFrame:
    path = out_dir / "runs.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def _read_aggregate_table(out_dir: Path) -> pd.DataFrame:
    path = out_dir / "aggregate_metrics.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def write_chinese_aggregate_summary(out_dir: Path) -> Path:
    summary = _read_aggregate_table(out_dir)
    runs = _read_runs_table(out_dir)
    lines = [
        "# GA/SA 特徵篩選中文彙整報告",
        "",
        "## 實驗設定",
        "",
        "- 資料集：IEEE-CIS Fraud Detection 的 train_transaction.csv。",
        "- GA/SA 搜尋：使用 20,000 筆 stratified proxy rows。",
        "- 最終評估：搜尋完成後，將特徵子集套用到完整 train_transaction.csv，使用 5-fold CV 重新評估。",
        "- 前處理：缺失值填補、類別欄位 factorize、VarianceThreshold、mutual information Top 150。",
        "- Fitness：AUC - lambda * selected_features / total_features。",
        "- 實際 preset 與參數：請以各 run 的 run_metadata.json 與 runs.csv 為準；word_strict 為完全嚴格版，word_practical/word_balanced 為加速版。",
        "- Baselines：Random Search、AllSearchFeatures、TopK_MI。",
        "",
        "## Word 要求檢查",
        "",
        "- 背景與研究問題：已涵蓋 fraud detection 特徵篩選與模型效能/特徵數取捨。",
        "- GA/SA 方法描述：程式保留 Word strict 參數；若本次 run 使用 word_practical/word_balanced，需在報告中註明搜尋階段為加速設定。",
        "- Baseline：包含 Random Search、AllSearchFeatures、TopK_MI。",
        "- 穩定性：multi-seed 彙整包含 mean/std 與 GA/SA Jaccard overlap。",
        "- 圖表：四張報告用圖在 figures_for_report/。",
        "",
        "## Multi-seed Runs",
        "",
        "```text",
        runs.to_string(index=False) if not runs.empty else "尚未產生 runs.csv",
        "```",
        "",
        "## Aggregate Metrics",
        "",
        "```text",
        summary.to_string(index=False) if not summary.empty else "尚未產生 aggregate_metrics.csv",
        "```",
    ]
    overlap_path = out_dir / "feature_overlap_summary.csv"
    if overlap_path.exists():
        overlap = pd.read_csv(overlap_path)
        lines.extend(["", "## GA/SA 特徵重疊", "", "```text", overlap.to_string(index=False), "```"])
    lines.extend([
        "",
        "## 報告檔案",
        "",
        "- aggregate_metrics.csv：各 lambda/method 的平均與標準差。",
        "- all_metrics.csv：所有 run 的完整 metric row。",
        "- feature_overlap_all_runs.csv：每個 seed 的 GA/SA Jaccard overlap。",
        "- figures_for_report/：Word 報告需要的四張圖。",
        "- GA_SA_report_summary_zh_TW.html：中文 HTML 彙整。",
    ])
    path = out_dir / "aggregate_summary_zh_TW.md"
    path.write_text("\n".join(lines), encoding="utf-8")

    fig_lines = [
        "# Word 報告圖表索引",
        "",
        "1. figures_for_report/01_convergence.png：GA/SA convergence。",
        "2. figures_for_report/02_feature_importance_selection.png：MI rank 與 GA/SA 選取。",
        "3. figures_for_report/03_pareto_auc_vs_features.png：AUC 與特徵數 Pareto。",
        "4. figures_for_report/04_lambda_sensitivity.png：lambda sensitivity。",
    ]
    (out_dir / "word_figures_zh_TW.md").write_text("\n".join(fig_lines), encoding="utf-8")
    return path


def _img_data_uri(path: Path) -> str:
    if not path.exists():
        return ""
    return "data:image/png;base64," + base64.b64encode(path.read_bytes()).decode("ascii")


def _table_html(df: pd.DataFrame, max_rows: int = 80) -> str:
    if df.empty:
        return "<p>尚未產生資料表。</p>"
    return df.head(max_rows).to_html(index=False, escape=True, border=0, classes="data-table")


def write_chinese_html_summary(out_dir: Path, representative_run: Path | None = None) -> Path:
    summary = _read_aggregate_table(out_dir)
    runs = _read_runs_table(out_dir)
    overlap = pd.read_csv(out_dir / "feature_overlap_summary.csv") if (out_dir / "feature_overlap_summary.csv").exists() else pd.DataFrame()
    all_metrics = pd.read_csv(out_dir / "all_metrics.csv") if (out_dir / "all_metrics.csv").exists() else pd.DataFrame()
    all_metrics_display = all_metrics.drop(columns=["mask_bits"], errors="ignore")

    best_lines = []
    if not summary.empty:
        full_seed_count = int(summary["n_runs"].max())
        for lam, group in summary.groupby("lambda", sort=True):
            comparable = group[group["n_runs"] == full_seed_count]
            if comparable.empty:
                comparable = group
            best_auc = comparable.sort_values("final_cv_auc_mean", ascending=False).iloc[0]
            best_fit = comparable.sort_values("final_cv_fitness_mean", ascending=False).iloc[0]
            best_lines.append(
                f"<li>lambda={lam}: 平均 AUC 最佳為 <b>{html.escape(str(best_auc['method']))}</b> "
                f"({best_auc['final_cv_auc_mean']:.4f} ± {best_auc['final_cv_auc_std']:.4f})；"
                f"平均 fitness 最佳為 <b>{html.escape(str(best_fit['method']))}</b> "
                f"({best_fit['final_cv_fitness_mean']:.4f} ± {best_fit['final_cv_fitness_std']:.4f})。</li>"
            )

    fig_dir = out_dir / "figures_for_report"
    figs = [
        ("GA/SA convergence", fig_dir / "01_convergence.png", "觀察 GA 與 SA 在不同 lambda 下的 fitness 收斂走勢。"),
        ("Feature importance / selection", fig_dir / "02_feature_importance_selection.png", "比較 mutual information 前段特徵與 GA/SA 實際選取結果。"),
        ("Pareto: AUC vs selected features", fig_dir / "03_pareto_auc_vs_features.png", "呈現模型 AUC 與特徵數量之間的取捨。"),
        ("Lambda sensitivity", fig_dir / "04_lambda_sensitivity.png", "比較 lambda 提高時 AUC 與選取特徵數的變化。"),
    ]
    fig_html = []
    for title, path, desc in figs:
        uri = _img_data_uri(path)
        if uri:
            fig_html.append(f"<figure><img src='{uri}' alt='{html.escape(title)}'><figcaption><b>{html.escape(title)}</b>：{html.escape(desc)}</figcaption></figure>")
        else:
            fig_html.append(f"<p>缺少圖檔：{html.escape(str(path))}</p>")

    generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    html_text = f"""<!doctype html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>GA/SA 特徵篩選中文彙整報告</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Noto Sans TC", Arial, sans-serif; margin: 0; color: #1f2933; background: #f7f8fa; line-height: 1.62; }}
main {{ max-width: 1120px; margin: 0 auto; padding: 32px 24px 56px; }}
h1 {{ font-size: 30px; margin: 0 0 8px; }}
h2 {{ font-size: 22px; margin-top: 34px; border-bottom: 1px solid #d8dee9; padding-bottom: 8px; }}
section {{ background: #fff; border: 1px solid #e3e7ee; border-radius: 8px; padding: 20px; margin: 18px 0; }}
.meta {{ color: #52606d; }}
ul {{ padding-left: 22px; }}
img {{ width: 100%; height: auto; border: 1px solid #dde3ec; border-radius: 6px; background: white; }}
figure {{ margin: 18px 0 28px; }}
figcaption {{ margin-top: 8px; color: #3e4c59; }}
.data-table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
.data-table th, .data-table td {{ border: 1px solid #e1e7ef; padding: 6px 8px; text-align: right; }}
.data-table th:first-child, .data-table td:first-child {{ text-align: left; }}
.data-table th {{ background: #edf2f7; }}
.note {{ background: #eef6ff; border-left: 4px solid #2563eb; padding: 10px 12px; }}
</style>
</head>
<body>
<main>
<h1>GA/SA 特徵篩選最佳整合版中文彙整報告</h1>
<p class="meta">產生時間：{generated_at}<br>輸出資料夾：{html.escape(str(out_dir))}</p>
<section>
<h2>Word 需求對照</h2>
<ul>
<li>背景與研究問題：以 IEEE-CIS fraud detection 做特徵篩選，評估 AUC、特徵數、穩定性與執行時間。</li>
<li>資料與前處理：讀取 train_transaction.csv；GA/SA 搜尋使用 20,000 筆 stratified proxy rows；final CV 使用完整 train_transaction.csv。</li>
<li>方法與 fitness：比較 GA、SA、Random Search、AllSearchFeatures、Top-K MI，fitness = AUC - lambda * selected_ratio。</li>
<li>參數設定：實際 preset 與參數寫在 run_metadata.json 與 runs.csv；word_strict 完全對應 Word 表格，word_practical/word_balanced 是加速版。</li>
<li>實驗比較：輸出 final CV AUC、fitness、selected_count、時間、multi-seed mean/std、GA/SA Jaccard overlap。</li>
<li>圖表：包含 convergence、feature importance/selection、Pareto、lambda sensitivity 四張圖。</li>
</ul>
<p class="note">若老師要求搜尋 fitness 也完全照 Word 表格，請使用 word_strict；本加速版預設 word_practical，final evaluation 仍使用完整資料 5-fold，但搜尋階段是加速設定，報告中需誠實註明。</p>
</section>
<section>
<h2>實驗摘要</h2>
<ul>
{''.join(best_lines) if best_lines else '<li>尚未產生 aggregate_metrics.csv。</li>'}
</ul>
</section>
<section>
<h2>執行的 Runs</h2>
{_table_html(runs)}
</section>
<section>
<h2>Aggregate Metrics</h2>
{_table_html(summary)}
</section>
<section>
<h2>GA/SA 特徵重疊</h2>
{_table_html(overlap)}
</section>
<section>
<h2>報告圖表</h2>
{''.join(fig_html)}
</section>
<section>
<h2>完整 Metrics</h2>
<p class="note">Word 沒有要求報告列出 mask_bits；mask_bits 是程式內部用來重建 binary feature mask 的原始紀錄，完整欄位仍保留在 all_metrics.csv。</p>
{_table_html(all_metrics_display, max_rows=120)}
</section>
</main>
</body>
</html>"""
    path = out_dir / "GA_SA_report_summary_zh_TW.html"
    path.write_text(html_text, encoding="utf-8")
    return path


def latest_aggregate_dir(results_dir: Path = RESULTS_DIR) -> Path | None:
    candidates = [p for p in results_dir.glob("aggregate_*") if p.is_dir()]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None


def latest_run_dir(results_dir: Path = RESULTS_DIR) -> Path | None:
    candidates = [p for p in results_dir.glob("run_*") if p.is_dir()]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None


def show_latest_results(results_dir: Path = RESULTS_DIR) -> None:
    """Display paths and the latest aggregate table inside Jupyter."""
    out_dir = latest_aggregate_dir(results_dir)
    if out_dir is None:
        print("目前沒有 aggregate_* 結果。")
        return
    print("最新彙整資料夾：", out_dir)
    html_path = out_dir / "GA_SA_report_summary_zh_TW.html"
    if html_path.exists():
        print("中文 HTML：", html_path)
    metrics_path = out_dir / "aggregate_metrics.csv"
    if metrics_path.exists():
        try:
            from IPython.display import display
            display(pd.read_csv(metrics_path))
        except Exception:
            print(pd.read_csv(metrics_path).to_string(index=False))

## 開始執行

下一格預設就是完整 Word 需求版。若只要測試環境，先改開關。


In [ ]:
# ===== 這格是主要開關 =====
# 預設使用加速但仍保留核心報告輸出的 word_practical。
# 如果要完全照 Word 表格，改成 REAL_PRESET = "word_strict"，但會非常久。
# 如果只想先測環境，請改成 RUN_SMOKE=True、RUN_MULTI_SEED=False。

RUN_SMOKE = False
RUN_SINGLE_REAL = False
RUN_MULTI_SEED = True

REAL_PRESET = "word_practical"   # word_practical / word_balanced / word_strict
SEARCH_FEATURES = 150
PROXY_ROWS = 20_000
FINAL_EVAL_FULL_TRAINING_SET = True
SEARCH_DEVICE = "cpu"             # 搜尋階段會反覆訓練小模型；本機舊結果顯示 CPU 較快
FINAL_DEVICE = "auto"             # final full-data CV 較大，偵測到 NVIDIA 時可用 GPU
SEEDS = [7, 42, 2026]             # 3 個 seed 才能看穩定性與 std；趕時間可先改 [7, 42]

if RUN_SMOKE:
    run_single_experiment(
        mode="synthetic",
        preset="smoke",
        device="cpu",
        proxy_rows=3_000,
        search_features=30,
        seed=42,
        final_eval_full_training_set=False,
    )

if RUN_SINGLE_REAL:
    run_single_experiment(
        mode="real",
        preset=REAL_PRESET,
        data_path=DATA_PATH,
        device=SEARCH_DEVICE,
        final_device=FINAL_DEVICE,
        proxy_rows=PROXY_ROWS,
        search_features=SEARCH_FEATURES,
        seed=42,
        final_eval_full_training_set=FINAL_EVAL_FULL_TRAINING_SET,
    )

if RUN_MULTI_SEED:
    run_multi_seed_experiment(
        seeds=SEEDS,
        mode="real",
        preset=REAL_PRESET,
        data_path=DATA_PATH,
        device=SEARCH_DEVICE,
        final_device=FINAL_DEVICE,
        search_features=SEARCH_FEATURES,
        proxy_rows=PROXY_ROWS,
        final_eval_full_training_set=FINAL_EVAL_FULL_TRAINING_SET,
    )

if not any([RUN_SMOKE, RUN_SINGLE_REAL, RUN_MULTI_SEED]):
    print("目前三個執行開關都是 False，所以沒有開始訓練。")
    print("要跑加速整合版：把 RUN_MULTI_SEED 改成 True，REAL_PRESET 用 word_practical，然後執行這格。")


## 檢視最新結果

跑完後執行下一格。


In [ ]:
# 跑完後執行這格，可以看到最新 aggregate table 與中文 HTML 路徑。
show_latest_results()